In [ ]:
from datasets import load_from_disk
from transformers import (
    AutoTokenizer,
    AutoModelForCausalLM,
    DataCollatorForSeq2Seq,
    TrainingArguments,
    Trainer,
    BitsAndBytesConfig
)
from peft import LoraConfig, get_peft_model
import torch

In [ ]:
# 加载基础模型
base_model_path = "/mnt/project/Qwen2.5-1.5B-lora/model/Qwen2.5-1.5B"

# 加载分词器
tokenizer = AutoTokenizer.from_pretrained(
    base_model_path,
    max_seq_length=2048,
    padding_side="right",
    use_fast=True
)

# 确保设置 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载模型
base_model = AutoModelForCausalLM.from_pretrained(
    base_model_path,
    torch_dtype=torch.bfloat16,
    use_cache=False,
)

In [ ]:
# 加载基础模型
lora_model_path = "/mnt/project/Qwen2.5-1.5B-lora/output/Qwen2.5-1.5B-lora"

# 确保设置 pad_token
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

# 加载模型
lora_model = AutoModelForCausalLM.from_pretrained(
    lora_model_path,
    torch_dtype=torch.bfloat16,
    use_cache=False,
)

In [ ]:
test_dataset = load_from_disk("/mnt/project/Qwen2.5-1.5B-lora/data/test_dataset")

In [ ]:
test_dataset[3:5]

In [ ]:
prompt_style = """Below is an instruction that describes a task, paired with an input that provides further context. Write a response that appropriately completes the request.
Before answering, think carefully about the question and create a step-by-step chain of thoughts to ensure a logical and accurate response.

### Instruction:
You are a medical expert with advanced knowledge in clinical reasoning, diagnostics, and treatment planning. Please answer the following medical question.

### Question:
{}

### Response:
<think>{}"""


# 原始模型推理

In [ ]:
question = "一位72岁的男性患者，有20年的高血压病史，现在在全麻复合硬膜外麻醉下接受胆囊手术后，哪种镇痛方法最为合适？"

base_model.eval()
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = base_model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1024,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])

# 原始模型推理

In [ ]:
lora_model.eval()
inputs = tokenizer([prompt_style.format(question, "")], return_tensors="pt").to("cuda")

outputs = lora_model.generate(
    input_ids=inputs.input_ids,
    attention_mask=inputs.attention_mask,
    max_new_tokens=1024,
    use_cache=True,
)

response = tokenizer.batch_decode(outputs)
print(response[0].split("### Response:")[1])